# # 06 — Grille décisionnelle et validation finale des hypothèses

### - Algorithme  (ordre : n → linéarité → interprétabilité)
### - H1 : données simulées documentées + résultat RMSE affiché
### - H2 : résultat chiffré affiché
### - H3 : protocole d'outliers documenté + dégradation 
### - Tableau synthétique des 4 hypothèses



In [ ]:


import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from sklearn.datasets import make_regression, fetch_california_housing

from src.hypotheses import test_h1, test_h2, test_h3



## 1. Test H1 — Linéarité

In [ ]:

print("╔══════════════════════════════════════════════════════╗")
print("║  H1 — Données linéaires simulées                    ║")
print("╠══════════════════════════════════════════════════════╣")
print("  Protocole :")
print("  - make_regression(n_samples=1000, n_features=5, noise=0.1)")
print("  - noise=0.1 → relation quasi-linéaire")
print("  - OLS vs Random Forest (n_estimators=100)")
print("  - H1 validée si RMSE_OLS ≤ RMSE_RF\n")

X_lin, y_lin = make_regression(
    n_samples=1000,
    n_features=5,
    noise=0.1,
    random_state=42
)

h1 = test_h1(X_lin, y_lin)

print()

╔══════════════════════════════════════════════════════╗
║  H1 — Données linéaires simulées                    ║
╠══════════════════════════════════════════════════════╣
  Protocole :
  - make_regression(n_samples=1000, n_features=5, noise=0.1)
  - noise=0.1 → relation quasi-linéaire
  - OLS vs Random Forest (n_estimators=100)
  - H1 validée si RMSE_OLS ≤ RMSE_RF

=== H1 — Linéarité (données simulées) ===
RMSE OLS           : 0.1053
RMSE Random Forest : 16.8904
H1 = True
Conclusion : H1 validée (OLS ≤ RF)



## 2. Test H2 — Nécessité de modèles complexes (données réelles)

In [3]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Data
data = fetch_california_housing()
X = data.data
y = data.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Models
rf = RandomForestRegressor(n_estimators=100, random_state=42)
xgb = XGBRegressor(
    n_estimators=100,
    random_state=42,
    eval_metric="rmse"
)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# Predictions
rmse_rf = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb.predict(X_test)))

# Relative gain
gain = (rmse_rf - rmse_xgb) / rmse_rf * 100

# H2 condition
h2 = gain >= 10

print("=== H2 — California Housing ===")
print(f"RMSE RF  : {rmse_rf:.4f}")
print(f"RMSE XGB : {rmse_xgb:.4f}")
print(f"Gain     : {gain:.2f}%")
print(f"H2       : {h2}")

if h2:
    print("Conclusion : H2 validée (XGBoost améliore RF ≥ 10%)")
else:
    print("Conclusion : H2 rejetée")

=== H2 — California Housing ===
RMSE RF  : 0.5053
RMSE XGB : 0.4739
Gain     : 6.22%
H2       : False
Conclusion : H2 rejetée


## 3. Test H3 — Robustesse aux outliers (données réelles)

In [ ]:

print("╔══════════════════════════════════════════════════════╗")
print("║  H3 — Robustesse aux outliers                       ║")
print("╠══════════════════════════════════════════════════════╣")
print("  Protocole :")
print("  - California Housing (mêmes données que H2)")
print("  - Injection de 5% d'outliers dans y_train")
print("    (facteur multiplicatif aléatoire ∈ [10, 50])")
print("  - Comparaison des dégradations % de RMSE_test")
print("  - H3 validée si dégradation_RF < dégradation_XGB\n")

h3 = test_h3(X, y, outlier_fraction=0.05)
print()



╔══════════════════════════════════════════════════════╗
║  H3 — Robustesse aux outliers                       ║
╠══════════════════════════════════════════════════════╣
  Protocole :
  - California Housing (mêmes données que H2)
  - Injection de 5% d'outliers dans y_train
    (facteur multiplicatif aléatoire ∈ [10, 50])
  - Comparaison des dégradations % de RMSE_test
  - H3 validée si dégradation_RF < dégradation_XGB

=== H3 — Robustesse aux outliers ===
Outliers injectés : 825 (5%)
Facteur multiplicatif : ×[10, 50]

RMSE RF  : 0.5053 → 5.4143
RMSE XGB : 0.4739 → 6.4226

Dégradation RF  : 971.42%
Dégradation XGB : 1255.25%

H3 = True
Conclusion : H3 validée (RF plus robuste)



## 4. Tableau synthétique des hypothèses

In [ ]:

tableau_hypotheses = pd.DataFrame({
    "Hypothèse": ["H1", "H2", "H3"],
    "Énoncé": [
        "RF ≈ OLS sur données linéaires (Δ RMSE < 5%)",
        "XGBoost améliore RF de ≥ 10% (RMSE)",
        "RF plus robuste aux outliers que XGBoost"
    ],
    "Résultat": [
        f"{'True' if h1 else 'False'}",
        f"{'True' if h2  else 'False'}",
        f"{'True' if h3 else 'False'}"
    ],
    
    "Conclusion": [
        "Processus non-linéaire" if not h1 else "Processus linéaire",
        "Modèles complexes nécessaires" if h2 else "Amélioration insuffisante",
        "RF plus robuste" if h3 else "XGB plus robuste"
    ]
})

print("\n=== Tableau — Synthèse des hypothèses ===")
print(tableau_hypotheses.to_string(index=False))




=== Tableau — Synthèse des hypothèses ===
Hypothèse                                       Énoncé Résultat                Conclusion
       H1 RF ≈ OLS sur données linéaires (Δ RMSE < 5%)     True        Processus linéaire
       H2          XGBoost améliore RF de ≥ 10% (RMSE)    False Amélioration insuffisante
       H3     RF plus robuste aux outliers que XGBoost     True           RF plus robuste


## 5. Ordre de vérification : n → linéarité → interprétabilité

In [ ]:

def recommander_modele(n_samples, linearite, besoin_interpretabilite,
                       tolerance_risque='medium'):
    """
    Recommande un modèle selon la grille décisionnelle (Tableau 4.7).

    Parameters
    ----------
    n_samples                : int   — taille du dataset
    linearite                : str   — 'forte' ou 'faible'
    besoin_interpretabilite  : str   — 'élevé', 'moyen', 'faible'
    tolerance_risque         : str   — 'high', 'medium', 'low'

    Ordre de vérification : n → linéarité → interprétabilité
    """
    # ── Petit dataset (n < 500) ───────────────────────────────────
    if n_samples < 500:
        if linearite == 'forte':
            return "Régression Linéaire  [n<500, linéaire]"
        else:
            return "Random Forest  [n<500, non-linéaire]"

    # ── Dataset moyen (500 ≤ n < 5000) ────────────────────────────
    elif n_samples < 5000:
        if linearite == 'forte':
            if besoin_interpretabilite == 'élevé':
                return "Régression Linéaire  [n moyen, linéaire, interp. élevée]"
            else:
                return "Random Forest  [n moyen, linéaire, interp. faible]"
        else:
            if besoin_interpretabilite == 'élevé':
                return "Random Forest + SHAP  [n moyen, non-linéaire, interp. élevée]"
            else:
                return "Random Forest  [n moyen, non-linéaire]"

    # ── Grand dataset (n ≥ 5000) ──────────────────────────────────
    else:
        if linearite == 'forte' and besoin_interpretabilite == 'élevé':
            return "Régression Linéaire  [n>5000, linéaire, interp. élevée]"

        elif linearite == 'forte' and besoin_interpretabilite != 'élevé':
            return "Régression Linéaire ou RF  [n>5000, linéaire]"

        elif linearite == 'faible' and besoin_interpretabilite == 'élevé':
            # CORRECTION : RF+SHAP (pas OLS) pour non-linéaire + interprétabilité
            return "Random Forest + SHAP  [n>5000, non-linéaire, interp. élevée]"

        elif linearite == 'faible':
            if tolerance_risque == 'high':
                return "XGBoost  [n>5000, non-linéaire, risque toléré]"
            elif tolerance_risque == 'medium':
                return "Random Forest  [n>5000, non-linéaire, risque modéré]"
            else:
                return "Random Forest  [n>5000, non-linéaire, risque faible]"

        else:
            return "Random Forest  [cas par défaut]"




## ── Tests de la grille ────────────────────────────────────────────

In [ ]:

print("\n=== Tests de la grille décisionnelle ===")
scenarios = [
    (300,   'forte',  'élevé',  'low'),
    (2000,  'faible', 'élevé',  'medium'),
    (20640, 'faible', 'moyen',  'low'),      # California Housing
    (20640, 'faible', 'moyen',  'high'),
    (20640, 'faible', 'élevé',  'medium'),   # non-linéaire + interp. → RF+SHAP (pas OLS)
    (20640, 'forte',  'élevé',  'low'),
    (8000,  'faible', 'faible', 'high'),
]

for n, lin, interp, risk in scenarios:
    rec = recommander_modele(n, lin, interp, risk)
    print(f"  n={n:6d} | lin={lin:6s} | interp={interp:6s} | risk={risk:6s}  →  {rec}")



=== Tests de la grille décisionnelle ===
  n=   300 | lin=forte  | interp=élevé  | risk=low     →  Régression Linéaire  [n<500, linéaire]
  n=  2000 | lin=faible | interp=élevé  | risk=medium  →  Random Forest + SHAP  [n moyen, non-linéaire, interp. élevée]
  n= 20640 | lin=faible | interp=moyen  | risk=low     →  Random Forest  [n>5000, non-linéaire, risque faible]
  n= 20640 | lin=faible | interp=moyen  | risk=high    →  XGBoost  [n>5000, non-linéaire, risque toléré]
  n= 20640 | lin=faible | interp=élevé  | risk=medium  →  Random Forest + SHAP  [n>5000, non-linéaire, interp. élevée]
  n= 20640 | lin=forte  | interp=élevé  | risk=low     →  Régression Linéaire  [n>5000, linéaire, interp. élevée]
  n=  8000 | lin=faible | interp=faible | risk=high    →  XGBoost  [n>5000, non-linéaire, risque toléré]
